# VPC Operations with Boto3

Region: ap-south-1

In [ ]:
import boto3
from botocore.exceptions import ClientError

REGION = "ap-south-1"
DRY_RUN = True

ec2 = boto3.client("ec2", region_name=REGION)

In [ ]:
VPC_CIDR = "10.20.0.0/16"
PUBLIC_SUBNET_CIDR = "10.20.1.0/24"
AVAILABILITY_ZONE = "ap-south-1a"

In [ ]:
def maybe(action, func, **kwargs):
    if DRY_RUN:
        print(f"[DRY RUN] {action}: {kwargs}")
        return None
    try:
        return func(**kwargs)
    except ClientError as exc:
        print(f"Error during {action}: {exc}")
        return None

In [ ]:
if DRY_RUN:
    vpc_id = "vpc-PLACEHOLDER"
    subnet_id = "subnet-PLACEHOLDER"
    igw_id = "igw-PLACEHOLDER"
    route_table_id = "rtb-PLACEHOLDER"
    association_id = "rtbassoc-PLACEHOLDER"
    sg_id = "sg-PLACEHOLDER"
else:
    vpc = ec2.create_vpc(CidrBlock=VPC_CIDR)
    vpc_id = vpc["Vpc"]["VpcId"]
    ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={"Value": True})
    ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={"Value": True})

    subnet = ec2.create_subnet(VpcId=vpc_id, CidrBlock=PUBLIC_SUBNET_CIDR, AvailabilityZone=AVAILABILITY_ZONE)
    subnet_id = subnet["Subnet"]["SubnetId"]
    ec2.modify_subnet_attribute(SubnetId=subnet_id, MapPublicIpOnLaunch={"Value": True})

    igw = ec2.create_internet_gateway()
    igw_id = igw["InternetGateway"]["InternetGatewayId"]
    ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)

    route_table = ec2.create_route_table(VpcId=vpc_id)
    route_table_id = route_table["RouteTable"]["RouteTableId"]
    ec2.create_route(RouteTableId=route_table_id, DestinationCidrBlock="0.0.0.0/0", GatewayId=igw_id)
    association = ec2.associate_route_table(RouteTableId=route_table_id, SubnetId=subnet_id)
    association_id = association["AssociationId"]

    sg = ec2.create_security_group(GroupName="exam-sg", Description="SSH and HTTP", VpcId=vpc_id)
    sg_id = sg["GroupId"]
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[
            {"IpProtocol": "tcp", "FromPort": 22, "ToPort": 22, "IpRanges": [{"CidrIp": "0.0.0.0/0"}]},
            {"IpProtocol": "tcp", "FromPort": 80, "ToPort": 80, "IpRanges": [{"CidrIp": "0.0.0.0/0"}]}
        ]
    )

print("VPC ID:", vpc_id)
print("Subnet ID:", subnet_id)
print("IGW ID:", igw_id)
print("Route Table ID:", route_table_id)
print("Security Group ID:", sg_id)

In [ ]:
if DRY_RUN:
    print("[DRY RUN] cleanup: delete security group, disassociate route table, delete route table, detach and delete IGW, delete subnet, delete VPC")
else:
    ec2.delete_security_group(GroupId=sg_id)
    if association_id:
        ec2.disassociate_route_table(AssociationId=association_id)
    ec2.delete_route_table(RouteTableId=route_table_id)
    ec2.detach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
    ec2.delete_internet_gateway(InternetGatewayId=igw_id)
    ec2.delete_subnet(SubnetId=subnet_id)
    ec2.delete_vpc(VpcId=vpc_id)